In [ ]:
import torch
import torch.optim as optim
from matplotlib import pyplot as plt
from torch.utils.data import DataLoader

# Dataset

In [ ]:
import os
import logging
from PIL import Image
import torch
import torch.utils.data as data
from pycocotools.coco import COCO
from torchvision import transforms
from collections import defaultdict
import numpy as np
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True  # Handle truncated image files

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class MyCOCODataset(data.Dataset):
    def __init__(self, data_root, annofile, output_size=(224, 224), is_train=True, transform=None):
        """
        Initialize COCO dataset
        Args:
            data_root (str): Image root directory
            annofile (str): Annotation file path
            output_size (tuple): Output image size
            is_train (bool): Whether in training mode
            transform (torchvision.transforms.Compose, optional): Custom transforms
        """
        self.data_root = data_root
        self.annofile = annofile
        self.is_train = is_train
        self.output_size = output_size
        self.image_cache = {}  # Add image cache

        # Check paths
        if not os.path.exists(data_root):
            raise FileNotFoundError(f"Data root not found: {data_root}")
        if not os.path.exists(annofile):
            raise FileNotFoundError(f"Annotation file not found: {annofile}")

        # Initialize COCO
        self.coco = COCO(annofile)
        self.all_ann_ids = list(self.coco.anns.keys())
        self.all_anns = self.coco.anns

        # Get all categories and create mappings
        self.category_ids = sorted(self.coco.getCatIds())
        self.num_classes = len(self.category_ids)
        self.catid_to_idx = {cat_id: idx for idx, cat_id in enumerate(self.category_ids)}
        self.idx_to_catid = {idx: cat_id for cat_id, idx in self.catid_to_idx.items()}
        self.categories = [self.coco.cats[cat_id]['name'] for cat_id in self.category_ids]

        logger.info(f"Number of categories: {self.num_classes}")
        logger.info(f"Categories: {self.categories}")

        # Cache image paths
        self.img_id_to_path = {}
        for img in self.coco.loadImgs(self.coco.getImgIds()):
            img_path = os.path.join(self.data_root, img['file_name'])
            self.img_id_to_path[img['id']] = img_path

        # Pre-filter invalid samples and calculate class distribution
        self._filter_samples()

        # Count samples for each class
        self._print_class_distribution()

        # Calculate class weights
        self.class_weights = self._calculate_class_weights()
        logger.info(f"Class weights: {self.class_weights}")

        # Set data augmentation
        self.transform = transform if transform else self._get_default_transforms()

    def _get_default_transforms(self):
        """Get default data augmentation strategy"""
        if self.is_train:
            return transforms.Compose([
                # Basic adjustments
                transforms.Resize((int(self.output_size[0]*1.2), int(self.output_size[1]*1.2))),
                transforms.RandomResizedCrop(
                    self.output_size, 
                    scale=(0.7, 1.0),
                    ratio=(0.75, 1.333)
                ),
                
                # Geometric transformations
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomVerticalFlip(p=0.2),
                transforms.RandomRotation(15),
                transforms.RandomAffine(
                    degrees=10, 
                    translate=(0.1, 0.1),
                    scale=(0.9, 1.1)
                ),
                
                # Color transformations
                transforms.ColorJitter(
                    brightness=0.3,
                    contrast=0.3,
                    saturation=0.3,
                    hue=0.1
                ),
                transforms.RandomAutocontrast(p=0.2),
                transforms.RandomEqualize(p=0.2),
                
                # Blur and noise
                transforms.RandomApply([
                    transforms.GaussianBlur(kernel_size=3)
                ], p=0.1),
                
                # Normalization
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]
                ),
                transforms.RandomErasing(p=0.2)
            ])
        else:
            return transforms.Compose([
                transforms.Resize(self.output_size),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]
                )
            ])

    def _filter_samples(self):
        """Filter invalid samples and calculate class distribution"""
        self.valid_instance_ids = []
        self.class_counts = defaultdict(int)

        logger.info("Starting to filter invalid samples...")
        total_samples = len(self.all_ann_ids)
        valid_count = 0

        for ann_id in self.all_ann_ids:
            ann = self.all_anns[ann_id]
            img_id = ann["image_id"]
            category_id = ann["category_id"]
            mapped_label = self.catid_to_idx.get(category_id, None)

            if mapped_label is None:
                continue

            img_path = self.img_id_to_path.get(img_id, None)
            if img_path is None or not os.path.exists(img_path):
                continue

            try:
                img = Image.open(img_path)
                if img.mode != "RGB":
                    img = img.convert("RGB")
                if min(img.size) < 32:  # Minimum size threshold
                    continue
                
                # Check bbox validity
                bbox = ann.get("bbox", [])
                if len(bbox) != 4 or not all(isinstance(x, (int, float)) for x in bbox):
                    continue
                
                self.valid_instance_ids.append(ann_id)
                self.class_counts[mapped_label] += 1
                valid_count += 1
                
                # Print progress periodically
                if valid_count % 1000 == 0:
                    logger.info(f"Processed {valid_count}/{total_samples} samples...")
                
            except Exception as e:
                continue

        logger.info(f"Filtering completed. Valid instances: {len(self.valid_instance_ids)}")

    def _print_class_distribution(self):
        """Print sample distribution for each class"""
        total_samples = sum(self.class_counts.values())
        logger.info("\n=== Class Distribution ===")
        for idx, count in self.class_counts.items():
            category_name = self.categories[idx]
            percentage = (count / total_samples) * 100 if total_samples > 0 else 0
            logger.info(f"{category_name:15s}: {count:5d} samples ({percentage:5.1f}%)")
        logger.info("==========================\n")

    def _calculate_class_weights(self):
        """Calculate class weights to handle class imbalance"""
        total_samples = sum(self.class_counts.values())
        if total_samples == 0:
            logger.warning("Total samples is zero. Using uniform weights.")
            return torch.ones(self.num_classes)

        weights = []
        # Use frequency-based weight calculation
        frequencies = np.array([self.class_counts.get(i, 0) for i in range(self.num_classes)])
        frequencies = np.where(frequencies > 0, frequencies, 1)  # Avoid division by zero
        weights = np.median(frequencies) / frequencies
        weights = np.clip(weights, 0.1, 10.0)  # Limit weight range

        return torch.tensor(weights, dtype=torch.float)

    def _validate_and_load_image(self, img_path):
        """Validate and load image, using cache"""
        if img_path in self.image_cache:
            return self.image_cache[img_path].copy()

        if not os.path.exists(img_path):
            raise FileNotFoundError(f"Image not found: {img_path}")

        try:
            img = Image.open(img_path)
            if img.mode != "RGB":
                img = img.convert("RGB")

            # Validate image size
            if min(img.size) < 32:
                raise ValueError(f"Image too small: {img.size}")

            # Cache image (limit cache size)
            if len(self.image_cache) < 1000:  # Cache maximum 1000 images
                self.image_cache[img_path] = img.copy()

            return img
        except Exception as e:
            logger.error(f"Error loading image {img_path}: {str(e)}")
            raise

    def _process_bbox(self, bbox, img_size):
        """Process and validate bounding box"""
        x, y, w, h = bbox

        # Ensure bbox is within image bounds
        x = max(0, min(x, img_size[0]-1))
        y = max(0, min(y, img_size[1]-1))
        w = min(w, img_size[0]-x)
        h = min(h, img_size[1]-y)

        # Handle invalid bbox
        if w < 10 or h < 10:
            # Return bbox for entire image
            return [0, 0, img_size[0], img_size[1]]

        # Add bbox context
        context = 0.2
        dx = int(w * context)
        dy = int(h * context)
        
        x = max(0, x - dx)
        y = max(0, y - dy)
        w = min(img_size[0] - x, w + 2*dx)
        h = min(img_size[1] - y, h + 2*dy)

        return [x, y, w, h]

    def _crop_and_process_image(self, img, bbox):
        """Crop and process image"""
        try:
            # Process bbox
            bbox = self._process_bbox(bbox, img.size)
            x, y, w, h = map(int, bbox)

            # Crop image
            cropped = img.crop((x, y, x+w, y+h))

            # Validate crop result
            if min(cropped.size) < 32:
                return img

            return cropped
        except Exception as e:
            logger.warning(f"Error in crop_and_process_image: {str(e)}")
            return img

    def __getitem__(self, index):
        """Get one sample from dataset"""
        ann_id = self.valid_instance_ids[index]
        ann = self.all_anns[ann_id]
        img_id = ann["image_id"]
        bbox = ann["bbox"]
        category_id = ann["category_id"]
        mapped_label = self.catid_to_idx[category_id]

        # Load and process image
        img_path = self.img_id_to_path[img_id]
        img = self._validate_and_load_image(img_path)
        processed_img = self._crop_and_process_image(img, bbox)

        # Apply data augmentation
        if self.transform:
            transformed_img = self.transform(processed_img)
        else:
            transformed_img = transforms.ToTensor()(processed_img)

        # Get class weight
        weight = self.class_weights[mapped_label]

        return transformed_img, torch.tensor(mapped_label, dtype=torch.long), weight

    def __len__(self):
        """Return dataset size"""
        return len(self.valid_instance_ids)

    def get_class_names(self):
        """Get all class names"""
        return {idx: name for idx, name in enumerate(self.categories)}

    def get_dataset_stats(self):
        """Get dataset statistics"""
        stats = {
            'num_classes': self.num_classes,
            'class_distribution': dict(self.class_counts),
            'total_samples': len(self.valid_instance_ids),
            'class_weights': self.class_weights.tolist(),
            'categories': self.categories
        }
        return stats

if __name__ == "__main__":
    # Test dataset
    try:
        data_root = "C:/Users/28100/Desktop/AI/MLproject/ML-Project_Autlab/2024_uestc_autlab/coco"
        annofile = "C:/Users/28100/Desktop/AI/MLproject/ML-Project_Autlab/2024_uestc_autlab/coco/annotations.json"

        # Create dataset
        dataset = MyCOCODataset(
            data_root=data_root,
            annofile=annofile,
            is_train=True
        )

        # Test data loading
        logger.info("\nTesting data loading...")
        for i in range(min(5, len(dataset))):
            try:
                img, label, weight = dataset[i]
                logger.info(f"\nSample {i}:")
                logger.info(f"Image shape: {img.shape}")
                logger.info(f"Label: {label.item()}")
                logger.info(f"Weight: {weight.item()}")
                class_names = dataset.get_class_names()
                logger.info(f"Category name: {class_names.get(label.item(), 'Unknown')}")
            except Exception as e:
                logger.error(f"Error loading sample {i}: {str(e)}")

        # Print dataset statistics
        stats = dataset.get_dataset_stats()
        logger.info("\nDataset Statistics:")
        logger.info(f"Total samples: {stats['total_samples']}")
        logger.info(f"Number of classes: {stats['num_classes']}")
        logger.info("Class distribution:")
        for idx, count in stats['class_distribution'].items():
            category = stats['categories'][idx]
            logger.info(f"{category:15s}: {count:5d} samples")
            
        # Test data augmentation consistency
        logger.info("\nTesting data augmentation consistency...")
        test_sample, _, _ = dataset[0]
        augmented_samples = []
        for _ in range(3):
            aug_sample, _, _ = dataset[0]
            augmented_samples.append(aug_sample)
        
        # Check differences between augmented images (verify randomness)
        differences = [torch.abs(augmented_samples[i] - augmented_samples[i-1]).mean().item() 
                      for i in range(1, len(augmented_samples))]
        logger.info(f"Average difference between augmented samples: {np.mean(differences):.4f}")
        
        # Test bbox processing
        logger.info("\nTesting bbox processing...")
        ann_id = dataset.valid_instance_ids[0]
        ann = dataset.all_anns[ann_id]
        img_id = ann["image_id"]
        img_path = dataset.img_id_to_path[img_id]
        original_bbox = ann["bbox"]
        img = dataset._validate_and_load_image(img_path)
        processed_bbox = dataset._process_bbox(original_bbox, img.size)
        logger.info(f"Original bbox: {original_bbox}")
        logger.info(f"Processed bbox: {processed_bbox}")

    except Exception as e:
        logger.error(f"Dataset initialization error: {str(e)}")
        import traceback
        traceback.print_exc()

# Model (Here you can change the deep learning model)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import logging

logger = logging.getLogger(__name__)

class SEBlock(nn.Module):
    """Squeeze-and-Excitation Block"""
    def __init__(self, channels, reduction=16):
        super(SEBlock, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class ImprovedModel(nn.Module):
    def __init__(self, num_classes=7, model_name='efficientnet_b0', dropout_rate=0.5):
        super(ImprovedModel, self).__init__()
        self.model_name = model_name
        self.num_classes = num_classes
        self.dropout_rate = dropout_rate

        # Load pretrained model
        if model_name == 'efficientnet_b0':
            self.base_model = models.efficientnet_b0(pretrained=True)
            num_ftrs = self.base_model.classifier[1].in_features
            # Keep features only
            self.base_model = self.base_model.features

            # Freeze early layers
            for param in self.base_model[:-4].parameters():
                param.requires_grad = False

        elif model_name == 'resnet50':
            self.base_model = models.resnet50(pretrained=True)
            num_ftrs = self.base_model.fc.in_features
            # Remove the final FC layer but keep conv layers
            self.base_model = nn.Sequential(*list(self.base_model.children())[:-2])

            # Freeze except last block
            for name, param in self.base_model.named_parameters():
                if 'layer4' not in name:
                    param.requires_grad = False
        else:
            raise ValueError(f"Unsupported model name: {model_name}")

        # Feature extractor
        self.feature_extractor = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(p=0.2),
            nn.Linear(num_ftrs, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate)
        )

        # Classifier
        self.classifier = nn.Linear(512, num_classes)

        # Initialize weights
        self._initialize_weights()
        
        # Print model info
        self._print_model_info(num_ftrs)

    def _initialize_weights(self):
        """Initialize weights"""
        for m in self.feature_extractor.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

        nn.init.xavier_normal_(self.classifier.weight)
        if self.classifier.bias is not None:
            nn.init.constant_(self.classifier.bias, 0)

    def _print_model_info(self, num_ftrs):
        """Print model information"""
        logger.info(f"\nModel Configuration:")
        logger.info(f"Base Model: {self.model_name}")
        logger.info(f"Number of classes: {self.num_classes}")
        logger.info(f"Feature dimension: {num_ftrs}")
        logger.info(f"Dropout rate: {self.dropout_rate}")
        
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        logger.info(f"Total parameters: {total_params:,}")
        logger.info(f"Trainable parameters: {trainable_params:,}")

    def forward(self, x):
        """Forward pass"""
        # Extract features - will maintain spatial dimensions
        features = self.base_model(x)
        # Process features
        features = self.feature_extractor(features)
        # Classify
        out = self.classifier(features)
        return out

    def get_features(self, x):
        """Get feature representations"""
        features = self.base_model(x)
        return self.feature_extractor(features)

    def unfreeze_layers(self, num_layers=3):
        """Unfreeze specified number of layers"""
        if self.model_name == 'efficientnet_b0':
            layers = list(self.base_model.children())
            for layer in layers[-num_layers:]:
                for param in layer.parameters():
                    param.requires_grad = True
        elif self.model_name == 'resnet50':
            layers_to_unfreeze = [f'layer{i}' for i in range(5-num_layers, 5)]
            for name, param in self.base_model.named_parameters():
                if any(layer in name for layer in layers_to_unfreeze):
                    param.requires_grad = True

        logger.info(f"Unfroze last {num_layers} layers")

    def get_layer_groups(self):
        """Get layer groups for differential learning rates"""
        if self.model_name == 'efficientnet_b0':
            return [
                {'params': list(self.base_model[-4:].parameters()), 'lr_mult': 0.1},
                {'params': self.feature_extractor.parameters(), 'lr_mult': 0.5},
                {'params': self.classifier.parameters(), 'lr_mult': 1.0}
            ]
        elif self.model_name == 'resnet50':
            return [
                {'params': self.base_model[-1].parameters(), 'lr_mult': 0.1},
                {'params': self.feature_extractor.parameters(), 'lr_mult': 0.5},
                {'params': self.classifier.parameters(), 'lr_mult': 1.0}
            ]

def create_model(model_config):
    """Factory function to create model"""
    model = ImprovedModel(
        num_classes=model_config.get('num_classes', 7),
        model_name=model_config.get('model_name', 'efficientnet_b0'),
        dropout_rate=model_config.get('dropout_rate', 0.5)
    )
    return model

if __name__ == "__main__":
    # Test model
    try:
        # Configure logging
        logging.basicConfig(level=logging.INFO)
        
        # Create model
        model_config = {
            'num_classes': 7,
            'model_name': 'efficientnet_b0',
            'dropout_rate': 0.5
        }
        model = create_model(model_config)

        # Test forward pass
        x = torch.randn(4, 3, 224, 224)  # batch size of 4
        output = model(x)
        logger.info(f"\nInput shape: {x.shape}")
        logger.info(f"Output shape: {output.shape}")

        # Test feature extraction
        features = model.get_features(x)
        logger.info(f"Feature shape: {features.shape}")

        # Test layer groups
        layer_groups = model.get_layer_groups()
        logger.info("\nLayer groups for optimizer:")
        for idx, group in enumerate(layer_groups):
            logger.info(f"Group {idx}: LR multiplier = {group['lr_mult']}")

    except Exception as e:
        logger.error(f"Model testing error: {str(e)}")
        import traceback
        traceback.print_exc()

# Trainining

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from model import create_model
from dataset import MyCOCODataset
import os
import time
from datetime import datetime
import logging
import pickle
from collections import defaultdict

def setup_logging(output_dir):
    """Set up logging"""
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(os.path.join(output_dir, 'training.log')),
            logging.StreamHandler()
        ]
    )

class AverageMeter(object):
    """Compute and store the average and current value"""
    def __init__(self, name, fmt=':f'):
        self.name = name
        self.fmt = fmt
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

    def __str__(self):
        fmtstr = '{name} {val' + self.fmt + '} ({avg' + self.fmt + '})'
        return fmtstr.format(**self.__dict__)

def create_data_loader(train_dataset, config):
    """Create data loader"""
    # Different configuration for Windows system
    is_windows = os.name == 'nt'
    num_workers = 0 if is_windows else min(8, os.cpu_count())
    persistent_workers = False if is_windows else True

    train_loader = DataLoader(
        train_dataset,
        batch_size=config['batch_size'],
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=False,
        persistent_workers=persistent_workers
    )

    return train_loader

def train_epoch(model, train_loader, criterion, optimizer, scaler, device, epoch, scheduler=None):
    """Train for one epoch"""
    model.train()
    losses = AverageMeter('Loss', ':.4e')
    top1 = AverageMeter('Acc@1', ':6.2f')
    batch_time = AverageMeter('Time', ':6.3f')
    
    # Track accuracy for each class
    class_correct = defaultdict(int)
    class_total = defaultdict(int)
    
    end = time.time()
    
    for i, (images, labels, weights) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)
        weights = weights.to(device)

        # Calculate batch weights
        batch_weights = weights / weights.sum()

        optimizer.zero_grad()

        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)  # [batch_size]
            weighted_loss = (loss * batch_weights).mean()  # Calculate weighted average loss

        # Gradient clipping
        scaler.scale(weighted_loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        scaler.step(optimizer)
        scaler.update()

        if scheduler is not None:
            scheduler.step()

        # Calculate accuracy
        _, predicted = outputs.max(1)
        correct = predicted.eq(labels)
        acc = correct.float().mean() * 100

        # Update class-wise accuracy
        for label, is_correct in zip(labels, correct):
            class_total[label.item()] += 1
            if is_correct:
                class_correct[label.item()] += 1

        # Update meters
        losses.update(weighted_loss.item(), images.size(0))
        top1.update(acc.item(), images.size(0))
        batch_time.update(time.time() - end)
        end = time.time()

        # Print training status
        if i % 10 == 0:
            logging.info(
                'Epoch: [{0}][{1}/{2}]\t'
                'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                'Acc@1 {top1.val:.3f} ({top1.avg:.3f})'.format(
                    epoch, i, len(train_loader), batch_time=batch_time,
                    loss=losses, top1=top1))

    # Output accuracy for each class
    logging.info("\nClass-wise Training Accuracies:")
    for cls in sorted(class_total.keys()):
        if class_total[cls] > 0:
            cls_acc = 100.0 * class_correct[cls] / class_total[cls]
            category_name = train_loader.dataset.dataset.categories[cls]
            logging.info(f'{category_name:15s}: {cls_acc:.2f}% ({class_correct[cls]}/{class_total[cls]})')

    return losses.avg, top1.avg

def save_checkpoint(state, is_best, output_dir, epoch):
    """Save checkpoint"""
    filename = os.path.join(output_dir, f'checkpoint_epoch_{epoch}.pth')
    torch.save(state, filename)
    if is_best:
        best_filename = os.path.join(output_dir, 'best_model.pth')
        torch.save(state, best_filename)

def main():
    # Configuration parameters
    config = {
        'data_root': "C:/Users/28100/Desktop/AI/MLproject/ML-Project_Autlab/2024_uestc_autlab/coco",
        'annofile': "C:/Users/28100/Desktop/AI/MLproject/ML-Project_Autlab/2024_uestc_autlab/coco/annotations.json",
        'lr': 3e-4,
        'min_lr': 1e-6,
        'batch_size': 32,
        'num_classes': 7,
        'num_epochs': 200,
        'weight_decay': 1e-4,
        'model_name': 'efficientnet_b0',
        'dropout_rate': 0.5,
        'output_size': (224, 224),
        'warmup_epochs': 5,
        'seed': 42
    }

    # Set random seed
    torch.manual_seed(config['seed'])
    if torch.cuda.is_available():
        torch.cuda.manual_seed(config['seed'])
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    # Create output directory
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_dir = f'outputs/run_{timestamp}'
    os.makedirs(output_dir, exist_ok=True)

    # Setup logging
    setup_logging(output_dir)
    
    # Set device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logging.info(f"Using device: {device}")

    # Load dataset
    dataset = MyCOCODataset(
        data_root=config['data_root'],
        annofile=config['annofile'],
        is_train=True,
        output_size=config['output_size']
    )

    # Split dataset (only create test set, not used in training)
    indices = list(range(len(dataset)))
    train_indices, test_indices = train_test_split(
        indices, test_size=0.2, stratify=[dataset[i][1].item() for i in indices],
        random_state=config['seed']
    )

    # Save test indices
    test_indices_path = os.path.join(output_dir, 'test_indices.pkl')
    with open(test_indices_path, 'wb') as f:
        pickle.dump(test_indices, f)
    logging.info(f"Test indices saved to {test_indices_path}")

    # Create training set
    train_dataset = Subset(dataset, train_indices)

    # Create data loader
    train_loader = create_data_loader(train_dataset, config)

    logging.info(f"\nDataset split:")
    logging.info(f"Training set: {len(train_dataset)} samples")
    logging.info(f"Test set: {len(test_indices)} samples")

    # Create model
    model = create_model(config)
    model = model.to(device)

    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)

    # Get layer groups for differential learning rates
    layer_groups = model.module.get_layer_groups() if isinstance(model, nn.DataParallel) else model.get_layer_groups()

    # Optimizer
    optimizer = optim.AdamW(layer_groups, lr=config['lr'], weight_decay=config['weight_decay'])

    # Learning rate scheduling
    num_training_steps = len(train_loader) * config['num_epochs']
    num_warmup_steps = len(train_loader) * config['warmup_epochs']

    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=config['lr'],
        total_steps=num_training_steps,
        pct_start=num_warmup_steps/num_training_steps,
        anneal_strategy='cos',
        final_div_factor=config['lr']/config['min_lr']
    )

    # Loss function
    class_weights = dataset.class_weights.to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights, reduction='none')

    # Mixed precision training
    scaler = GradScaler()

    # Training loop
    best_acc = 0
    best_epoch = 0
    early_stopping_counter = 0
    early_stopping_patience = 15

    try:
        for epoch in range(config['num_epochs']):
            logging.info(f"\nEpoch {epoch+1}/{config['num_epochs']}")
            logging.info('-' * 60)

            # Training phase
            train_loss, train_acc = train_epoch(
                model, train_loader, criterion, optimizer, scaler, device, epoch, scheduler
            )

            # Record learning rate
            current_lr = optimizer.param_groups[0]['lr']
            logging.info(f'Current LR: {current_lr:.6f}')
            logging.info(f'Training Loss: {train_loss:.4f}, Accuracy: {train_acc:.2f}%')

            # Check for best model
            is_best = train_acc > best_acc
            if is_best:
                best_acc = train_acc
                best_epoch = epoch
                early_stopping_counter = 0
                logging.info(f'New best model training accuracy: {best_acc:.2f}%')
            else:
                early_stopping_counter += 1
                logging.info(f'No improvement for {early_stopping_counter} epochs')

            # Save checkpoint
            if isinstance(model, nn.DataParallel):
                model_state_dict = model.module.state_dict()
            else:
                model_state_dict = model.state_dict()

            save_checkpoint({
                'epoch': epoch,
                'model_state_dict': model_state_dict,
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_acc': best_acc,
                'config': config
            }, is_best, output_dir, epoch)

            # Early stopping
            if early_stopping_counter >= early_stopping_patience:
                logging.info(f'Early stopping triggered after {early_stopping_patience} epochs without improvement')
                break

        logging.info(f'\nTraining completed!')
        logging.info(f'Best training accuracy: {best_acc:.2f}% at epoch {best_epoch+1}')

    except KeyboardInterrupt:
        logging.info('\nTraining interrupted by user')
        if best_acc > 0:
            logging.info(f'Best training accuracy achieved: {best_acc:.2f}% at epoch {best_epoch+1}')
    except Exception as e:
        logging.error(f'Training failed: {str(e)}')
        raise

if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        logging.error(f"Training failed: {str(e)}")
        import traceback
        traceback.print_exc()

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)
for epoch in range(num_epoch):  # loop over the dataset multiple times
    # trainning loop
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs; data is a list of [inputs, labels]
        images, labels = data

        # NOTE: the input data should be normalized to [0, 1]
        # or to distribution with mean 0 and std 1
        images = images.float() / 255.0

        # NOTE: the input shape for Pytorch should be [batch_size, channels, height, width]
        # so we need to transpose the input data
        images = images.permute(0, 3, 1, 2)

        images, labels = images.to(device), labels.to(device)

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        print(f"a batch done, batch id: {i} loss: {loss.item()}")

    # validation loop
    best_accuracy = 0
    validate_predicte = []
    validate_ground_truth = []
    with torch.no_grad():
        for data in val_loader:
            images, labels = data

            # data preprocessing
            images = images.float() / 255.0
            images = images.permute(0, 3, 1, 2)

            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, predictions = torch.max(outputs, 1)
            validate_predicte.append(predictions.detach())
            validate_ground_truth.append(labels.detach())

    # calulate the accuracy
    validate_predicte = torch.cat(validate_predicte)
    validate_ground_truth = torch.cat(validate_ground_truth)
    correct = (validate_predicte == validate_ground_truth).sum().item()
    accuracy = correct / len(validate_ground_truth)

    # save checkpoint if the accuracy is better
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        torch.save(model.state_dict(), "outputs/best_model.pth")

    print(f"epoch: {epoch} accuracy: {accuracy}")

# Evaluation (you can also change the confusion matrix colors and also add F1 score matrics)

In [ ]:
import os
import json
import logging
import pickle
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
from dataset import MyCOCODataset
from model import create_model

def setup_logging(output_dir):
    """Set up logging"""
    os.makedirs(output_dir, exist_ok=True)
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(os.path.join(output_dir, 'evaluation.log')),
            logging.StreamHandler()
        ]
    )

def create_data_loader(dataset, indices, batch_size):
    """Create data loader"""
    # Ensure all indices are within valid range
    valid_indices = [idx for idx in indices if idx < len(dataset)]
    logging.info(f"Original indices count: {len(indices)}")
    logging.info(f"Valid indices count: {len(valid_indices)}")
    logging.info(f"Dataset length: {len(dataset)}")
    
    if len(valid_indices) != len(indices):
        logging.warning(f"Removed {len(indices) - len(valid_indices)} invalid indices")
    
    if len(valid_indices) == 0:
        logging.error("No valid indices found!")
        raise ValueError("No valid indices for creating DataLoader")
    
    test_dataset = Subset(dataset, valid_indices)
    
    # Different configuration for Windows system
    is_windows = os.name == 'nt'
    num_workers = 0 if is_windows else min(8, os.cpu_count())
    persistent_workers = False if is_windows else True
    
    loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=persistent_workers
    )
    
    logging.info(f"Created DataLoader with {len(loader)} batches")
    return loader

def load_model(model_path, config, device):
    """Load model"""
    model = create_model(config)
    checkpoint = torch.load(model_path, map_location=device)
    
    # Check if DataParallel was used
    state_dict = checkpoint['model_state_dict']
    if list(state_dict.keys())[0].startswith('module.'):
        model = nn.DataParallel(model)
    
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()
    return model

def evaluate_model(model, test_loader, device):
    """Evaluate model"""
    all_preds = []
    all_labels = []
    total_correct = 0
    total_samples = 0
    
    try:
        with torch.no_grad():
            for i, batch in enumerate(test_loader):
                logging.info(f"Processing batch {i+1}/{len(test_loader)}")
                
                if len(batch) == 3:  # If batch contains weights
                    images, labels, _ = batch
                else:  # If batch only contains images and labels
                    images, labels = batch
                
                logging.info(f"Batch size: {images.size(0)}")
                logging.info(f"Labels in batch: {labels.tolist()}")
                
                images = images.to(device)
                labels = labels.to(device)
                
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                
                logging.info(f"Predictions in batch: {predicted.tolist()}")
                
                total_samples += labels.size(0)
                total_correct += (predicted == labels).sum().item()
                
                # Convert to numpy and store
                batch_preds = predicted.cpu().numpy()
                batch_labels = labels.cpu().numpy()
                
                all_preds.extend(batch_preds)
                all_labels.extend(batch_labels)
                
                # Record current batch accuracy
                batch_acc = 100 * (predicted == labels).sum().item() / labels.size(0)
                logging.info(f"Batch {i+1} accuracy: {batch_acc:.2f}%")
        
        logging.info(f"Evaluation completed. Total samples processed: {total_samples}")
        logging.info(f"Total predictions: {len(all_preds)}")
        logging.info(f"Total labels: {len(all_labels)}")
        
        if len(all_preds) == 0:
            logging.error("No predictions were generated!")
            
    except Exception as e:
        logging.error(f"Error during evaluation: {str(e)}")
        import traceback
        traceback.print_exc()
        return 0.0, [], []
    
    if total_samples == 0:
        logging.error("No samples were processed!")
        return 0.0, [], []
        
    accuracy = 100 * total_correct / total_samples
    return accuracy, all_preds, all_labels

def plot_confusion_matrix(conf_matrix, categories, output_dir):
    """Plot confusion matrix"""
    plt.figure(figsize=(12, 10))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=categories, yticklabels=categories)
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.xticks(rotation=45)
    plt.yticks(rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'confusion_matrix.png'))
    plt.close()

def main():
    # Configuration
    config = {
        'data_root': "C:/Users/28100/Desktop/AI/MLproject/ML-Project_Autlab/2024_uestc_autlab/coco",  # Change to COCO root directory
        'annofile': "C:/Users/28100/Desktop/AI/MLproject/ML-Project_Autlab/2024_uestc_autlab/coco/annotations.json",
        'model_name': 'efficientnet_b0',
        'num_classes': 7,
        'batch_size': 32,
        'output_size': (224, 224)
    }
    
    # Set up output directory and logging
    output_dir = 'evaluation_results'
    setup_logging(output_dir)
    
    try:
        # Set device
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        logging.info(f"Using device: {device}")
        
        # Check file paths
        logging.info("Checking file paths...")
        if not os.path.exists(config['data_root']):
            raise FileNotFoundError(f"Data root not found: {config['data_root']}")
        if not os.path.exists(config['annofile']):
            raise FileNotFoundError(f"Annotation file not found: {config['annofile']}")
            
        # Check images folder
        images_dir = os.path.join(config['data_root'], 'JPEGImages').replace('\\', '/')
        logging.info(f"Checking images directory: {images_dir}")
        if not os.path.exists(images_dir):
            raise FileNotFoundError(f"Images directory not found: {images_dir}")
            
        # List some image files as examples
        image_files = os.listdir(images_dir)[:5]
        logging.info(f"Sample image files: {image_files}")
        
        # Check annotations file
        with open(config['annofile'], 'r') as f:
            anno = json.load(f)
            logging.info(f"Annotations info:")
            logging.info(f"Number of images: {len(anno.get('images', []))}")
            logging.info(f"Number of annotations: {len(anno.get('annotations', []))}")
            logging.info(f"Number of categories: {len(anno.get('categories', []))}")
        
        logging.info("Loading dataset...")
        # Load training dataset
        train_dataset = MyCOCODataset(
            data_root=config['data_root'],
            annofile=config['annofile'],
            is_train=True,
            output_size=config['output_size']
        )
        logging.info(f"Training dataset size: {len(train_dataset)}")
        
        # Load test dataset
        dataset = MyCOCODataset(
            data_root=config['data_root'],
            annofile=config['annofile'],
            is_train=False,
            output_size=config['output_size']
        )
        logging.info(f"Test dataset size: {len(dataset)}")
        
        if len(dataset) == 0:
            raise ValueError("Dataset is empty! Please check the data loading process.")
            
        # Load test set indices
        test_indices_path = 'C:/Users/28100/Desktop/AI/MLproject/ML-Project_Autlab/2024_uestc_autlab/outputs/run_20250107_160049/test_indices.pkl'
        logging.info(f"Loading test indices from {test_indices_path}")
        
        with open(test_indices_path, 'rb') as f:
            test_indices = pickle.load(f)
        
        logging.info(f"Number of test indices: {len(test_indices)}")
        
        # Create data loader
        test_loader = create_data_loader(dataset, test_indices, config['batch_size'])
        logging.info(f"Created test loader with {len(test_loader)} batches")
        
        # Load model
        model_path = 'C:/Users/28100/Desktop/AI/MLproject/ML-Project_Autlab/2024_uestc_autlab/outputs/run_20250107_160049/best_model.pth'
        logging.info(f"Loading model from {model_path}")
        model = load_model(model_path, config, device)
        
        # Evaluate model
        logging.info("Starting evaluation...")
        accuracy, all_preds, all_labels = evaluate_model(model, test_loader, device)
        logging.info(f"\nTest Accuracy: {accuracy:.2f}%")
        
        # Get category names
        categories = dataset.categories
        logging.info(f"Categories: {categories}")
        
        # Check if predictions exist
        if len(all_preds) == 0 or len(all_labels) == 0:
            logging.error("No predictions or labels available for confusion matrix")
            return
            
        logging.info(f"Number of predictions: {len(all_preds)}")
        logging.info(f"Number of labels: {len(all_labels)}")
        logging.info(f"Label range: {min(all_labels)} to {max(all_labels)}")
        logging.info(f"Prediction range: {min(all_preds)} to {max(all_preds)}")
        
        # Calculate confusion matrix
        conf_matrix = confusion_matrix(all_labels, all_preds)
        logging.info(f"Confusion matrix shape: {conf_matrix.shape}")
        
        if conf_matrix.size > 0:  # Ensure confusion matrix is not empty
            # Plot confusion matrix
            plot_confusion_matrix(conf_matrix, categories, output_dir)
            logging.info(f"Confusion matrix saved to {output_dir}/confusion_matrix.png")
        else:
            logging.error("Empty confusion matrix generated")
        
        # Print detailed classification report
        report = classification_report(
            all_labels, all_preds,
            target_names=categories,
            digits=3
        )
        logging.info("\nClassification Report:\n" + report)
        
        # Calculate accuracy for each class
        class_correct = conf_matrix.diagonal()
        class_total = conf_matrix.sum(axis=1)
        logging.info("\nPer-class accuracy:")
        for i, (correct, total) in enumerate(zip(class_correct, class_total)):
            if total > 0:
                class_acc = 100.0 * correct / total
                logging.info(f'{categories[i]:15s}: {class_acc:.2f}% ({correct}/{total})')
        
    except Exception as e:
        logging.error(f"Evaluation failed: {str(e)}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()